# Simple Tools

The `simple.py` module defines `Tool`, a lightweight LangChain tool that directly wraps a synchronous function, an asynchronous coroutine, or both.

`Tool` is intended for single-input operations. Tools requiring multiple named arguments should generally use `StructuredTool`.

# Tool

`Tool` wraps a Python function or coroutine and exposes it through the LangChain tool and Runnable interfaces.

## Bases

- `BaseTool`

## Attributes

1. `description`: Stores a description of the tool and its purpose.
   * **Type:**
     ```python
     description: str = ""
     ```

2. `func`: Stores the synchronous function executed by the tool.
   * **Type:**
     ```python
     func: Callable[..., str] | None
     ```

3. `coroutine`: Stores the asynchronous function executed by the tool.
   * **Type:**
     ```python
     coroutine: Callable[
         ...,
         Awaitable[str]
     ] | None = None
     ```

### Properties

1. `args`: Returns the input arguments accepted by the tool.

   When `args_schema` is available, the schema properties inherited from `BaseTool` are returned. Otherwise, the tool exposes one string argument named `tool_input` for backward compatibility.

   * **Type:**
     ```python
     args: dict[str, Any]
     ```

### Methods

1. `ainvoke`: Executes the tool asynchronously through the Runnable interface.

   When no coroutine is configured, the synchronous `invoke` method is executed in an executor. Otherwise, asynchronous execution is delegated to `BaseTool`.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: str | dict[str, Any] | ToolCall, # Tool input or complete tool call
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional execution arguments
     ) -> Any
     ```

2. `_to_args_and_kwargs`: Converts validated tool input into positional and keyword arguments.

   For backward compatibility, a simple `Tool` must receive exactly one input value. A `ToolException` is raised when multiple input values are supplied.

   * **Syntax:**
     ```python
     _to_args_and_kwargs(
         self,
         tool_input: str | dict[str, Any], # Input supplied to the tool
         tool_call_id: str | None # Identifier of the related tool call
     ) -> tuple[
         tuple[str, ...],
         dict[str, Any]
     ]
     ```

3. `_run`: Executes the configured synchronous function.

   When the function accepts a `callbacks` parameter, a child callback manager is injected. When the function accepts a Runnable configuration parameter, the current configuration is also injected.

   A `NotImplementedError` is raised when no synchronous function is configured.

   * **Syntax:**
     ```python
     _run(
         self,
         *args: Any, # Positional arguments passed to the function
         config: RunnableConfig, # Runtime configuration
         run_manager: CallbackManagerForToolRun | None = None, # Synchronous callback manager
         **kwargs: Any # Keyword arguments passed to the function
     ) -> Any
     ```

4. `_arun`: Executes the configured asynchronous coroutine.

   When the coroutine accepts a `callbacks` parameter, a child asynchronous callback manager is injected. When it accepts a Runnable configuration parameter, the current configuration is also injected.

   * **Syntax:**
     ```python
     async _arun(
         self,
         *args: Any, # Positional arguments passed to the coroutine
         config: RunnableConfig, # Runtime configuration
         run_manager: AsyncCallbackManagerForToolRun | None = None, # Asynchronous callback manager
         **kwargs: Any # Keyword arguments passed to the coroutine
     ) -> Any
     ```

5. `__init__`: Initializes a simple tool from its name, function, and description.

   This constructor is retained for backward compatibility.

   * **Syntax:**
     ```python
     __init__(
         self,
         name: str, # Name of the tool
         func: Callable[..., Any] | None, # Synchronous function executed by the tool
         description: str, # Description of the tool
         **kwargs: Any # Additional BaseTool fields
     ) -> None
     ```

6. `from_function`: Creates a `Tool` from a synchronous function, asynchronous coroutine, or both.

   A `ValueError` is raised when neither `func` nor `coroutine` is provided.

   * **Syntax:**
     ```python
     @classmethod
     from_function(
         cls,
         func: Callable[..., Any] | None, # Synchronous function to wrap
         name: str, # Name of the tool
         description: str, # Description of the tool
         return_direct: bool = False, # Whether to stop the agent loop after execution
         args_schema: ArgsSchema | None = None, # Optional input schema
         coroutine: Callable[
             ...,
             Awaitable[Any]
         ] | None = None, # Asynchronous function to wrap
         **kwargs: Any # Additional Tool fields
     ) -> Tool
     ```